In [3]:
import json
from pathlib import Path
from copy import deepcopy

src = Path("/mnt/data/yolo11_training.ipynb")
dst = Path("/mnt/data/yolo11_training_final_v2.ipynb")

with src.open("r", encoding="utf-8") as f:
    nb = json.load(f)

new_cells = [
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "# YOLO11 FINAL TRAINING V2\n",
            "\n",
            "This section starts a fresh final training run using the existing Apple-only YOLO dataset. Previous experiments are kept unchanged."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# ============================================================\n",
            "# FINAL V2 - VERIFY DATASET AND BASE MODEL\n",
            "# ============================================================\n",
            "\n",
            "from pathlib import Path\n",
            "from ultralytics import YOLO\n",
            "\n",
            "PROJECT_ROOT = Path(r\"C:\\major-project\")\n",
            "YOLO_ROOT = PROJECT_ROOT / \"model\" / \"yolo11\"\n",
            "YOLO_DATASET = PROJECT_ROOT / \"model\" / \"yolo11_dataset\"\n",
            "DATA_YAML = YOLO_DATASET / \"data.yaml\"\n",
            "BASE_MODEL = YOLO_ROOT / \"yolo11n.pt\"\n",
            "\n",
            "print(\"Dataset:\", YOLO_DATASET)\n",
            "print(\"Dataset exists:\", YOLO_DATASET.exists())\n",
            "print(\"data.yaml exists:\", DATA_YAML.exists())\n",
            "print(\"Base YOLO11 model:\", BASE_MODEL)\n",
            "print(\"Base model exists:\", BASE_MODEL.exists())\n",
            "\n",
            "assert YOLO_DATASET.exists(), \"YOLO dataset not found.\"\n",
            "assert DATA_YAML.exists(), \"data.yaml not found.\"\n",
            "assert BASE_MODEL.exists(), \"yolo11n.pt not found.\"\n            "
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# ============================================================\n",
            "# FINAL V2 - VERIFY TRAIN / VAL / TEST COUNTS\n",
            "# ============================================================\n",
            "\n",
            "for split in [\"train\", \"val\", \"test\"]:\n",
            "    image_dir = YOLO_DATASET / \"images\" / split\n",
            "    label_dir = YOLO_DATASET / \"labels\" / split\n",
            "\n",
            "    image_count = sum(1 for p in image_dir.iterdir() if p.is_file())\n",
            "    label_count = len(list(label_dir.glob(\"*.txt\")))\n",
            "\n",
            "    print(f\"{split.upper():5} -> {image_count} images | {label_count} labels\")\n",
            "\n",
            "    assert image_count == label_count, (\n",
            "        f\"{split}: image/label count mismatch\"\n",
            "    )\n",
            "\n",
            "print(\"\\nDataset verification passed.\")"
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# ============================================================\n",
            "# FINAL V2 - TRAIN YOLO11\n",
            "# ============================================================\n",
            "\n",
            "model = YOLO(str(BASE_MODEL))\n",
            "\n",
            "training_results = model.train(\n",
            "    data=str(DATA_YAML),\n",
            "    epochs=100,\n",
            "    imgsz=640,\n",
            "    batch=4,\n",
            "    patience=20,\n",
            "    device=\"cpu\",\n",
            "    workers=0,\n",
            "    pretrained=True,\n",
            "    cache=False,\n",
            "    plots=True,\n",
            "    save=True,\n",
            "    save_period=10,\n",
            "    project=str(YOLO_ROOT / \"runs\"),\n",
            "    name=\"apple_detection_final_v2\",\n",
            "    exist_ok=False\n",
            ")\n",
            "\n",
            "print(\"\\nFINAL V2 TRAINING COMPLETED.\")"
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# ============================================================\n",
            "# FINAL V2 - LOCATE BEST MODEL\n",
            "# ============================================================\n",
            "\n",
            "FINAL_RUN = YOLO_ROOT / \"runs\" / \"apple_detection_final_v2\"\n",
            "FINAL_BEST = FINAL_RUN / \"weights\" / \"best.pt\"\n",
            "FINAL_LAST = FINAL_RUN / \"weights\" / \"last.pt\"\n",
            "\n",
            "print(\"Run directory:\", FINAL_RUN)\n",
            "print(\"best.pt exists:\", FINAL_BEST.exists())\n",
            "print(\"last.pt exists:\", FINAL_LAST.exists())\n",
            "\n",
            "assert FINAL_BEST.exists(), \"best.pt was not created.\"\n            "
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# ============================================================\n",
            "# FINAL V2 - TEST SET EVALUATION\n",
            "# ============================================================\n",
            "\n",
            "final_model = YOLO(str(FINAL_BEST))\n\n",
            "metrics = final_model.val(\n",
            "    data=str(DATA_YAML),\n",
            "    split=\"test\",\n",
            "    imgsz=640,\n",
            "    batch=4,\n",
            "    device=\"cpu\",\n",
            "    workers=0,\n",
            "    plots=True,\n",
            "    project=str(YOLO_ROOT / \"runs\"),\n",
            "    name=\"apple_detection_final_v2_test\",\n",
            "    exist_ok=True\n",
            ")\n\n",
            "precision = float(metrics.box.mp)\n",
            "recall = float(metrics.box.mr)\n",
            "map50 = float(metrics.box.map50)\n",
            "map5095 = float(metrics.box.map)\n\n",
            "print(\"=\" * 60)\n",
            "print(\"YOLO11 FINAL V2 TEST PERFORMANCE\")\n",
            "print(\"=\" * 60)\n",
            "print(f\"Precision   : {precision * 100:.2f}%\")\n",
            "print(f\"Recall      : {recall * 100:.2f}%\")\n",
            "print(f\"mAP@50      : {map50 * 100:.2f}%\")\n",
            "print(f\"mAP@50-95   : {map5095 * 100:.2f}%\")\n",
            "print(\"=\" * 60)\n"
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# ============================================================\n",
            "# FINAL V2 - PERFORMANCE VISUALIZATION\n",
            "# ============================================================\n",
            "\n",
            "import matplotlib.pyplot as plt\n\n",
            "metric_names = [\n",
            "    \"Precision\",\n",
            "    \"Recall\",\n",
            "    \"mAP@50\",\n",
            "    \"mAP@50-95\"\n",
            "]\n\n",
            "metric_values = [\n",
            "    precision * 100,\n",
            "    recall * 100,\n",
            "    map50 * 100,\n",
            "    map5095 * 100\n",
            "]\n\n",
            "plt.figure(figsize=(10, 6))\n",
            "bars = plt.bar(metric_names, metric_values)\n",
            "plt.title(\"YOLO11 Apple Detection - Final V2 Performance\")\n",
            "plt.ylabel(\"Score (%)\")\n",
            "plt.ylim(0, 100)\n\n",
            "for bar, value in zip(bars, metric_values):\n",
            "    plt.text(\n",
            "        bar.get_x() + bar.get_width() / 2,\n",
            "        value + 1,\n",
            "        f\"{value:.2f}%\",\n",
            "        ha=\"center\"\n",
            "    )\n\n",
            "plt.tight_layout()\n",
            "plt.show()\n"
        ]
    }
]

nb2 = deepcopy(nb)
nb2.setdefault("cells", []).extend(new_cells)

with dst.open("w", encoding="utf-8") as f:
    json.dump(nb2, f, ensure_ascii=False, indent=1)

print(f"Created: {dst}")
print(f"Original cells: {len(nb.get('cells', []))}")
print(f"New cells added: {len(new_cells)}")
print(f"Total cells: {len(nb2.get('cells', []))}")


FileNotFoundError: [Errno 2] No such file or directory: '\\mnt\\data\\yolo11_training.ipynb'